# 03 · Memory — one memory, grown from many datasets

`build.py` grew one memory incrementally: it cognified an `encyclopedia` (Wikipedia) dataset, then *added* a `news` (CC-News) dataset — the graph grew, no rebuild. Here we query that unified memory; a single search draws on **both** datasets. (Wikipedia chunks start with `# Title`; news chunks are plain text.)

> Run `build.py` first.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # examples/demos
from _common import config
config.require_openai_key(); config.quiet()          # quiet cognee's verbose logs
config.configure("cognee_memory")
import cognee
from cognee.modules.search.types import SearchType
from cognee.infrastructure.databases.graph import get_graph_engine
from collections import Counter
g = await get_graph_engine()
m = await g.get_graph_metrics(include_optional=False)
print("unified memory:", m["num_nodes"], "nodes,", m["num_edges"], "edges")

unified memory: 1841 nodes, 5367 edges


## A single query retrieves from both datasets

In [2]:
def origin(t): return "encyclopedia" if str(t).lstrip().startswith("#") else "news"
hits = await config.search(query_text="notable people, places, and recent events",
                           query_type=SearchType.CHUNKS)
seen = Counter()
for h in (hits or [])[:8]:
    text = h.get("text", "") if isinstance(h, dict) else str(h)
    seen[origin(text)] += 1
    print(f"   [{origin(text)}] {text.strip().splitlines()[0][:80]}")
print("retrieved from:", dict(seen))

   [news] What's going on in ballet this week? We've pulled together some highlights.
   [encyclopedia] # Academy Awards
   [news] The Broadway revival of Richard Rogers and Oscar Hammerstein's Carousel opened l
   [encyclopedia] # Austin (disambiguation)
   [news] Jarrod Dicker, CEO of Po.et, talked with Dan Patterson about how his company use
   [news] Google Fiber showed new life in 2017, after a near death experience in late 2016
   [encyclopedia] # Ada
   [encyclopedia] # Academy Award for Best Production Design
retrieved from: {'news': 4, 'encyclopedia': 4}


## Answer over the whole memory

(cognee's `datasets=[...]` is a permission scope, not a retrieval filter — retrieval spans the unified memory; use a separate database per tenant for hard isolation.)

In [3]:
ans = await config.search(query_text="What kinds of topics does this memory cover?",
                          query_type=SearchType.GRAPH_COMPLETION)
print((ans[0] if isinstance(ans,(list,tuple)) and ans else ans))

The memory covers topics related to AI applications in business (cognitive era), agriculture (including food classes), IT security (AWS Certified Security exam), and educational use of technology (Windows 10 Lean). It also addresses exam content for AWS training and certifications.


## `node_set` — pull back just the slice each dataset tagged

Each `cognee.add(..., node_set=[tag])` tags its nodes. `get_nodeset_subgraph` extracts that slice of the unified memory — the nodes carrying the tag, plus their neighbours and the edges among them.

In [4]:
from cognee.modules.engine.models.node_set import NodeSet
for tag in ("reference", "current_events"):
    ns_nodes, ns_edges = await g.get_nodeset_subgraph(NodeSet, [tag])
    print(f"  '{tag}': {len(ns_nodes)} nodes, {len(ns_edges)} edges")

  'reference': 875 nodes, 2294 edges


  'current_events': 627 nodes, 1683 edges
